In [7]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, StaleElementReferenceException


def get_links_under_heading(driver, heading):
    try:
        heading_element = driver.find_element(By.XPATH, f"//h2[text()='{heading}']/following-sibling::ul")
        return [link.get_attribute('href') for link in heading_element.find_elements(By.TAG_NAME, 'a')]
    except:
        return []

# Set up Chrome service
service = ChromeService()

# Initialize Chrome WebDriver
driver = webdriver.Chrome(service=service)

# Open the main webpage
driver.get("https://www.servicecanada.gc.ca/tbsc-fsco/sc-hme.jsp")

# Wait for the elements to load
wait = WebDriverWait(driver, 10)
regions_section = wait.until(EC.presence_of_element_located((By.XPATH, "//h2[text()='Browse by Province or Territory']/following-sibling::ul")))

# Find the regional links
region_links = regions_section.find_elements(By.TAG_NAME, "a")
regional_urls = [link.get_attribute('href') for link in region_links]

# Dictionary to hold links for each region
all_region_links = {}

for url in regional_urls:
    driver.get(url)
    wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))

    try:
        # Wait for all links to be present on the page
        all_links = wait.until(EC.presence_of_all_elements_located((By.TAG_NAME, 'a')))

        # Filter links containing 'Canada Centre' or 'Scheduled' and exclude links with a dash in the text (only passport services)
        filtered_links = []
        for link in all_links:
            try:
                link_text = link.text
                if ('Canada Centre' in link_text or 'Scheduled' in link_text) and '-' not in link_text:
                    filtered_links.append(link.get_attribute('href'))
            except StaleElementReferenceException:
                # Re-find the element in the DOM
                continue

        all_region_links[url] = filtered_links

    except Exception as e:
        print(f"An error occurred while processing {url}: {e}")

# Close the browser
driver.quit()

# Close the browser
driver.quit()

# Print the collected links
for region, links in all_region_links.items():
    print(f"Links for {region}:")
    for link in links:
        print(link)
    print("\n")


Links for https://www.servicecanada.gc.ca/tbsc-fsco/sc-lst.jsp?prov=AB&lang=eng:
https://www.servicecanada.gc.ca/tbsc-fsco/sc-dsp.jsp?rc=4865&lang=eng
https://www.servicecanada.gc.ca/tbsc-fsco/sc-dsp.jsp?rc=4754&lang=eng
https://www.servicecanada.gc.ca/tbsc-fsco/sc-dsp.jsp?rc=4820&lang=eng
https://www.servicecanada.gc.ca/tbsc-fsco/sc-dsp.jsp?rc=4825&lang=eng
https://www.servicecanada.gc.ca/tbsc-fsco/sc-dsp.jsp?rc=4819&lang=eng
https://www.servicecanada.gc.ca/tbsc-fsco/sc-dsp.jsp?rc=4860&lang=eng
https://www.servicecanada.gc.ca/tbsc-fsco/sc-dsp.jsp?rc=4807&lang=eng
https://www.servicecanada.gc.ca/tbsc-fsco/sc-dsp.jsp?rc=4808&lang=eng
https://www.servicecanada.gc.ca/tbsc-fsco/sc-dsp.jsp?rc=4824&lang=eng
https://www.servicecanada.gc.ca/tbsc-fsco/sc-dsp.jsp?rc=4875&lang=eng
https://www.servicecanada.gc.ca/tbsc-fsco/sc-dsp.jsp?rc=4804&lang=eng
https://www.servicecanada.gc.ca/tbsc-fsco/sc-dsp.jsp?rc=4803&lang=eng
https://www.servicecanada.gc.ca/tbsc-fsco/sc-dsp.jsp?rc=4810&lang=eng
https://w

In [10]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
import pandas as pd
import time

# Initialize the Chrome WebDriver
service = ChromeService()
driver = webdriver.Chrome(service=service)
wait = WebDriverWait(driver, 10)

# Assuming all_region_links is a dictionary with regions as keys and list of links as values
timeout_links = {region: [(link, 0) for link in links] for region, links in all_region_links.items()}  # Pair each link with a retry counter

addresses = []  # List to store addresses

while timeout_links:
    new_timeout_links = {}  # Dictionary to keep track of links with TimeoutException in this iteration

    for region, links in timeout_links.items():
        for link, retry_count in links:
            try:
                driver.get(link)
                # It's better to use an explicit wait here instead of sleep if possible.
                # e.g., wait.until(EC.visibility_of_element_located((By.CLASS_NAME, "col-xs-10")))
                time.sleep(2)  # Wait for the page to load
                div_element = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "col-xs-10")))
                div_text = div_element.text
                addresses.append(div_text)
            except TimeoutException:
                print(f"Timed out waiting for page to load at link: {link}.")
                if retry_count < 2:  # Retry up to three times
                    if region not in new_timeout_links:
                        new_timeout_links[region] = []
                    new_timeout_links[region].append((link, retry_count + 1))  # Increment the retry counter
                else:
                    print(f"Skipping link after third attempt: {link}")
            except NoSuchElementException:
                print(f"The div element was not found on the page at link: {link}")
            except Exception as e:
                print(f"An error occurred at link {link}: {e}")

    timeout_links = new_timeout_links  # Update the links for the next iteration
    print("NEXT ROUND")


Timed out waiting for page to load at link: https://www.servicecanada.gc.ca/tbsc-fsco/sc-dsp.jsp?rc=4807&lang=eng.
Timed out waiting for page to load at link: https://www.servicecanada.gc.ca/tbsc-fsco/sc-dsp.jsp?rc=4808&lang=eng.
Timed out waiting for page to load at link: https://www.servicecanada.gc.ca/tbsc-fsco/sc-dsp.jsp?rc=1637&lang=eng.
Timed out waiting for page to load at link: https://www.servicecanada.gc.ca/tbsc-fsco/sc-dsp.jsp?rc=1613&lang=eng.
Timed out waiting for page to load at link: https://www.servicecanada.gc.ca/tbsc-fsco/sc-dsp.jsp?rc=NF-SAC&lang=eng.
Timed out waiting for page to load at link: https://www.servicecanada.gc.ca/tbsc-fsco/sc-dsp.jsp?rc=3601&lang=eng.
Timed out waiting for page to load at link: https://www.servicecanada.gc.ca/tbsc-fsco/sc-dsp.jsp?rc=4306&lang=eng.
NEXT ROUND
Timed out waiting for page to load at link: https://www.servicecanada.gc.ca/tbsc-fsco/sc-dsp.jsp?rc=1637&lang=eng.
NEXT ROUND
Timed out waiting for page to load at link: https://www.

In [11]:
import pandas as pd

# Save the addresses to an Excel file
df = pd.DataFrame(addresses, columns=['address'])
df.to_excel('can_pes.xlsx', index=False)

# Close the browser
driver.quit()
